In [ ]:
# ============================================================
# FULL CNN TRAINING CODE (TRAIN / VAL / TEST)
# Optimized to avoid slow loading / KeyboardInterrupt
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.optim.lr_scheduler import ReduceLROnPlateau

# =========================
# Device
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================
# Classes
# =========================
char_map = ['0','1','2','3','4','5','6','7','8','9','+','-','*','/','=']
num_classes = len(char_map)

# =========================
# Model
# =========================
class BestCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 , 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# =========================
# Transforms (LIGHT + FAST)
# =========================
train_tf = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

eval_tf = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# =========================
# Paths
# =========================
TRAIN_DIR = r"C:\Users\VICTUS\Desktop\MZN\num-data\train"
VAL_DIR   = r"C:\Users\VICTUS\Desktop\MZN\num-data\val"
TEST_DIR  = r"C:\Users\VICTUS\Desktop\MZN\num-data\test"

# =========================
# Datasets
# =========================
train_dataset = ImageFolder(TRAIN_DIR, transform=train_tf)
val_dataset   = ImageFolder(VAL_DIR,   transform=eval_tf)
test_dataset  = ImageFolder(TEST_DIR,  transform=eval_tf)

# =========================
# DataLoaders (FAST)
# =========================
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

# =========================
# Model Setup
# =========================
model = BestCNN(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="max", patience=3, factor=0.3)

# =========================
# Training Loop
# =========================
EPOCHS = 30
best_acc = 0.0

for epoch in range(EPOCHS):
    print(f"epochs:{epoch+1}|{EPOCHS}")
    model.train()
    running_loss = 0.0

    for i, (x, y) in enumerate(train_loader):
        
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        

       

    # ================= Validation =================
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            preds = model(x).argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    val_acc = correct / total
    scheduler.step(val_acc)

    print(
        f">>> Epoch {epoch+1} DONE | "
        f"Avg Loss: {running_loss/len(train_loader):.4f} | "
        f"Val Acc: {val_acc*100:.2f}%"
    )

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(
    {
        "state_dict": model.state_dict(),
        "classes": train_dataset.classes,  # <- ImageFolder true label order
    },
    "best_symbol_cnn2.pt"
)
        print(">>> Model saved")

# =========================
# Test Evaluation
# =========================
model.load_state_dict(torch.load("best_symbol_cnn2.pth"))
model.eval()

correct = 0
total = 0
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        preds = model(x).argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)

print("TEST ACCURACY:", 100 * correct / total)


Device: cuda
Train: 43195
Val: 5432
Test: 5453
epochs:1|30
>>> Epoch 1 DONE | Avg Loss: 0.1861 | Val Acc: 98.10%
>>> Model saved
epochs:2|30
>>> Epoch 2 DONE | Avg Loss: 0.0514 | Val Acc: 98.75%
>>> Model saved
epochs:3|30
>>> Epoch 3 DONE | Avg Loss: 0.0361 | Val Acc: 99.06%
>>> Model saved
epochs:4|30
>>> Epoch 4 DONE | Avg Loss: 0.0287 | Val Acc: 99.23%
>>> Model saved
epochs:5|30
>>> Epoch 5 DONE | Avg Loss: 0.0238 | Val Acc: 99.15%
epochs:6|30
>>> Epoch 6 DONE | Avg Loss: 0.0174 | Val Acc: 98.31%
epochs:7|30
>>> Epoch 7 DONE | Avg Loss: 0.0171 | Val Acc: 99.04%
epochs:8|30
>>> Epoch 8 DONE | Avg Loss: 0.0154 | Val Acc: 99.10%
epochs:9|30
>>> Epoch 9 DONE | Avg Loss: 0.0025 | Val Acc: 99.37%
>>> Model saved
epochs:10|30
>>> Epoch 10 DONE | Avg Loss: 0.0011 | Val Acc: 99.71%
>>> Model saved
epochs:11|30
>>> Epoch 11 DONE | Avg Loss: 0.0009 | Val Acc: 99.63%
epochs:12|30
>>> Epoch 12 DONE | Avg Loss: 0.0019 | Val Acc: 99.61%
epochs:13|30
>>> Epoch 13 DONE | Avg Loss: 0.0013 | Val Acc

C:\Users\VICTUS\AppData\Local\Temp\ipykernel_18472\2137327494.py:191: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_symbol_cnn2.pth"))

TEST ACCURACY: 99.7982761782505
